In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
from tqdm import tqdm
from xgboost import XGBClassifier

from catboost import CatBoostClassifier

from Utils.DataUtils import build_ae_dataloaders

from Models.FraudModel import FraudMLP
from Models.AutoEncoder import AutoEncoder
from Models.EnsembleModel import (
    TorchBinaryProbWrapper,
    WeightedFraudEnsemble,
    StackingFraudMLP,
    StackedFraudEnsemble,
    CatBoostAEWrapper,
    XGBoostAEWrapper,
    collect_member_probs,
    collect_weighted_ensemble_probs,
    collect_stacked_ensemble_probs,
    collect_labels,
)

from Utils.TrainUtils import get_device, train_model

In [2]:
train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


In [3]:
DEVICE = get_device()

# construct model ae256
autoencoder256 = AutoEncoder(
    input_dim=input_dim,
    latent_dim=256,
    hidden_dims=[128, 64, 32],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt256 = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)
ae_state_dict256 = ae_ckpt256["model_state_dict"] if "model_state_dict" in ae_ckpt256 else ae_ckpt256
autoencoder256.load_state_dict(ae_state_dict256)
autoencoder256.eval()

model_ae256 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    encoder=autoencoder256,
    freeze_encoder=True,
).to(DEVICE)

thr_cat = 0.57

cat_model = CatBoostClassifier()
cat_model.load_model("checkpoints/best_catboost.cbm")

cat_member = CatBoostAEWrapper(
    cat_model=cat_model,
    autoencoder=autoencoder256
)

In [4]:
xgb_model = XGBClassifier()
xgb_model.load_model("checkpoints/best_xgboost.json")

with open("checkpoints/best_xgboost_params.json", "r") as f:
    xgb_meta = json.load(f)

thr_xgb = xgb_meta["best_threshold"]

print("Loaded XGBoost threshold:", thr_xgb)
print("Loaded XGBoost params:", xgb_meta["params"])

xgb_member = XGBoostAEWrapper(
    xgb_model=xgb_model,
    autoencoder=autoencoder256,
)

Loaded XGBoost threshold: 0.09499999999999999
Loaded XGBoost params: {'iterations': 5000, 'depth': 10, 'learning_rate': 0.025}


In [5]:
@torch.no_grad()
def collect_single_member_probs(member, loader, device):
    all_probs = []
    for x, _ in loader:
        x = x.to(device)
        probs = member(x)
        all_probs.append(probs.cpu())
    return torch.cat(all_probs, dim=0)

In [6]:
cat_probs = collect_single_member_probs(cat_member, test_loader, DEVICE)
xgb_probs = collect_single_member_probs(xgb_member, test_loader, DEVICE)
test_labels = collect_labels(test_loader)

pred_cat = (cat_probs >= thr_cat)
pred_xgb = (xgb_probs >= thr_xgb)

disagreement = (pred_cat != pred_xgb).float().mean().item()

both_fraud = ((pred_cat == 1) & (pred_xgb == 1)).sum().item()
both_normal = ((pred_cat == 0) & (pred_xgb == 0)).sum().item()
cat_only = ((pred_cat == 1) & (pred_xgb == 0)).sum().item()
xgb_only = ((pred_cat == 0) & (pred_xgb == 1)).sum().item()

print("CatBoost vs XGBoost disagreement:", disagreement)
print("Both fraud :", both_fraud)
print("Both normal:", both_normal)
print("Cat only   :", cat_only)
print("XGB only   :", xgb_only)

CatBoost vs XGBoost disagreement: 0.011870491318404675
Both fraud : 1994
Both normal: 56359
Cat only   : 291
XGB only   : 410


In [7]:
y_true = test_labels

cat_only_mask = (pred_cat == 1) & (pred_xgb == 0)
xgb_only_mask = (pred_cat == 0) & (pred_xgb == 1)
both_fraud_mask = (pred_cat == 1) & (pred_xgb == 1)
either_mask = (pred_cat == 1) | (pred_xgb == 1)

print("Cat-only fraud rate :", y_true[cat_only_mask].float().mean().item())
print("XGB-only fraud rate :", y_true[xgb_only_mask].float().mean().item())
print("Both-fraud fraud rate:", y_true[both_fraud_mask].float().mean().item())
print("Either fraud rate   :", y_true[either_mask].float().mean().item())

Cat-only fraud rate : 0.10309278219938278
XGB-only fraud rate : 0.20000000298023224
Both-fraud fraud rate: 0.7487462162971497
Either fraud rate   : 0.5955473184585571
